In [1]:
import os
import re
import pandas as pd

In [2]:
RESULTS_DIR = "./results"
TARGET_GROUP = "03"
OUTPUT_EXCEL = os.path.join(
    RESULTS_DIR,
    f"Group_{TARGET_GROUP}_Analysis.xlsx"
)

In [3]:
def extract_accuracy(file_path):
    """
    Extrae Acc=XX.XX del archivo HResults.
    """
    with open(file_path, "r") as f:
        content = f.read()

    match = re.search(r'Acc=([0-9]+\.[0-9]+)', content)

    if match:
        return float(match.group(1))

    raise ValueError(f"Accuracy not found in {file_path}")

In [4]:
state_dirs = sorted([
    d for d in os.listdir(RESULTS_DIR)
    if re.match(r"results_\d+_states", d)
])

print("Detected models:")
for d in state_dirs:
    print(f"  {d}")

Detected models:
  results_10_states
  results_11_states
  results_12_states
  results_3_states
  results_4_states
  results_5_states
  results_6_states
  results_7_states
  results_8_states
  results_9_states


In [5]:
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:

    for state_dir in state_dirs:

        state_match = re.search(
            r"results_(\d+)_states",
            state_dir
        )

        states = state_match.group(1)

        results_root = os.path.join(
            RESULTS_DIR,
            state_dir
        )

        outer_group = f"Group_{TARGET_GROUP}"

        outer_path = os.path.join(
            results_root,
            outer_group
        )

        if not os.path.isdir(outer_path):

            print(
                f"Skipping {states} states "
                f"(Group_{TARGET_GROUP} not found)"
            )

            continue

        print(
            f"\nProcessing {states} states"
        )

        inner_groups = sorted([
            g for g in os.listdir(outer_path)
            if re.match(
                rf"Group{TARGET_GROUP}_\d+",
                g
            )
        ])

        if not inner_groups:
            continue

        gaussian_nums = set()

        for inner_group in inner_groups:

            inner_path = os.path.join(
                outer_path,
                inner_group
            )

            gaussian_dirs = [
                g for g in os.listdir(inner_path)
                if re.match(
                    r"\d+_gaussians",
                    g
                )
            ]

            for g in gaussian_dirs:

                gaussian_nums.add(
                    int(
                        re.search(
                            r"(\d+)_gaussians",
                            g
                        ).group(1)
                    )
                )

        gaussian_nums = sorted(
            gaussian_nums
        )

        df = pd.DataFrame(
            index=inner_groups,
            columns=[
                f"{g}_gaussians"
                for g in gaussian_nums
            ],
            dtype=float
        )

        for inner_group in inner_groups:

            inner_num = re.search(
                rf"Group{TARGET_GROUP}_(\d+)",
                inner_group
            ).group(1)

            inner_path = os.path.join(
                outer_path,
                inner_group
            )

            for g in gaussian_nums:

                results_file = os.path.join(
                    inner_path,
                    f"{g}_gaussians",
                    "HResults",
                    f"results{TARGET_GROUP}_state{inner_num}.txt"
                )

                if os.path.exists(results_file):

                    df.loc[
                        inner_group,
                        f"{g}_gaussians"
                    ] = extract_accuracy(
                        results_file
                    )

        df["mean"] = df.mean(axis=1)

        mean_row = df.mean(axis=0)

        df.loc["mean"] = mean_row

        df = df.round(2)

        sheet_name = f"{states}_states"

        df.to_excel(
            writer,
            sheet_name=sheet_name
        )

print(
    f"\nExcel generated correctly:\n"
    f"{OUTPUT_EXCEL}"
)


Processing 10 states

Processing 11 states

Processing 12 states

Processing 3 states

Processing 4 states

Processing 5 states

Processing 6 states

Processing 7 states

Processing 8 states

Processing 9 states

Excel generated correctly:
./results\Group_03_Analysis.xlsx
